# Agents and Retrieval Systems in LangChain Lab

[View Solution Notebook](https://static-assets.codecademy.com/Courses/Langchain_component/Module_3/Intelligent%20LangChain%20Agent%20with%20RAG.ipynb)

[Sample Course Document](https://static-assets.codecademy.com/Courses/Langchain_component/Module_3/sample_document.txt)

**Setup**

Run the cell below to import the necessary libraries and initialize the language model.

In [ ]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import TextLoader
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain import hub
import warnings
warnings.filterwarnings("ignore")

llm = ChatOpenAI(model="gpt-4o", temperature=0, stream=False)

**Note:** The OpenAI API key is already configured in this workspace. When working in an external environment, refer to the OpenAI API documentation for details on configuring your API key.


Be sure to run the Setup cell above before beginning with the tasks.

## Task Group 1 — Creating Tools

### Task 1

The assistant needs to handle calculation-based questions from learners. Language models cannot reliably perform arithmetic, so we create three dedicated math tools — `add()`, `subtract()`, and `multiply()` — that the agent can call instead of relying on the model's training.

The `@tool` decorator and function signatures are already provided.

+ Add a docstring to each function that describes when to use it.

+ Return the result of the corresponding arithmetic operation.


In [ ]:
@tool
def add(a: float, b: float) -> float:
    ### YOUR CODE HERE ###

@tool
def subtract(a: float, b: float) -> float:
    ### YOUR CODE HERE ###

@tool
def multiply(a: float, b: float) -> float:
    ### YOUR CODE HERE ###

Run the block below to verify each tool's name and description, and to test them with sample inputs.

In [ ]:
for t in [add, subtract, multiply]:
    print(f"Tool: {t.name} — {t.description}")

print(f"\nTest: 12 + 45 = {add.invoke({'a': 12, 'b': 45})}")
print(f"Test: 20 - 8 = {subtract.invoke({'a': 20, 'b': 8})}")
print(f"Test: 6 * 7 = {multiply.invoke({'a': 6, 'b': 7})}")

### Task 2

Learners also ask about recent developments in topics they are studying — information that is not in the course documentation and may change over time. We use `DuckDuckGoSearchRun`, already set up as `search_engine`, to give the assistant access to real-time information.

The `@tool` decorator defines a function named `web_search` with a single `str` parameter named `query`.


+ Add a docstring that describes what the tool does and when to use it — for current events or publicly available information.

+ Call `.run()` on `search_engine` with `query` and return the result.


In [ ]:
search_engine = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> str:
    ### YOUR CODE HERE ###

Run the block below to verify the tool's properties and test it with a sample query.

In [ ]:
print(f"Tool Name: {web_search.name}")
print(f"Tool Description: {web_search.description}")

result = web_search.invoke({"query": "LangChain framework"})
print(f"\nSearch Result Preview: {result[:100]}...")

## Task Group 2 — Building the Agent

## Task 3

The math and web search tools are ready. We now wire them into an agent that can reason about each learner question and decide which tool to call. The agent needs a structured prompt so it knows how to behave, where to receive input, and where to track its reasoning across tool calls.

Create a prompt template named `agent_prompt` with the following messages:

* a system message that describes the assistant as helpful and instructs it to use available tools
* a human message with an `{input}` placeholder
* a `MessagesPlaceholder` with `variable_name` set to `"agent_scratchpad"`


In [ ]:
tools = [add, subtract, multiply, web_search]

agent_prompt = ChatPromptTemplate.from_messages([
    ### YOUR CODE HERE ###
])

agent = create_tool_calling_agent(llm, tools, agent_prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, stream_runnable=False)

### Task 4

The agent is ready. We test it with two questions — one that requires a calculation and one that requires a web search — to verify that it selects the correct tool for each.

+ Invoke `agent_executor` with `"What is 124 plus 356?"` and store the result in `calc_result`.

+ Invoke `agent_executor` with `"What are the latest developments in AI this week?"` and store the result in `search_result`.

+ Uncomment the `print()` statements to display both answers.


In [ ]:
### YOUR CODE HERE ###
calc_result =
search_result =

# print("Calculation Result:", calc_result["output"])
# print("\nSearch Result:", search_result["output"])

**Note on Tool Selection:** The agent selects tools based on their docstrings alone. A clear, specific docstring leads to accurate tool selection — the agent does not need to be told explicitly which tool to use.



## Task Group 3 — Building the RAG Pipeline

### Task 5

The platform stores its [course documentation](https://static-assets.codecademy.com/Courses/Langchain_component/Module_3/sample_document.txt) in `sample_document.txt`. This file was never part of the language model's training data, so the assistant cannot answer questions about it from memory. To make it searchable, we break it into smaller chunks that can be embedded and retrieved individually.

+ Create a `TextLoader` instance using `file_path` and store it in `loader`.

+ Load the file contents and store the result in `documents`.

+ Create a text splitter instance named `text_splitter` with `chunk_size` set to `300` and `chunk_overlap` set to `50`.

+ Split `documents` and store the result in `chunks`.


In [ ]:
file_path = "sample_document.txt"

loader = ### YOUR CODE HERE ###
documents = ### YOUR CODE HERE ###

text_splitter = ### YOUR CODE HERE ###
chunks = ### YOUR CODE HERE ###

print(f"Original document: {len(documents[0].page_content)} characters")
print(f"After splitting: {len(chunks)} chunks")

### Task 6

The course documentation is now split into chunks. For the assistant to answer a learner's question like "What is a retriever?", it needs to find the most relevant chunks by meaning, not by matching exact words. We enable this by converting chunks into vectors and storing them in a searchable vector store.

+ Create an embedding model instance named `embeddings` using `"text-embedding-3-small"` as the model.

+ Create a vector store named `vector_store` from `chunks` and `embeddings`.


In [ ]:
embeddings = ### YOUR CODE HERE ###
vector_store = ### YOUR CODE HERE ###

print(f"Vectors stored: {vector_store.index.ntotal}")

## Task 7

The vector store is ready. The next step is to connect it to the language model so that when a learner asks a question, the assistant fetches the most relevant chunks and generates a grounded answer. A `retriever` and `rag_prompt` are already set up to support this.

+ Build a chain named `rag_chain` by mapping `"context"` to `retriever` and `"question"` to the user's input, then connect `rag_prompt`, `llm`, and an output parser in sequence.

+ Invoke `rag_chain` with `"What are the main components of LangChain?"` and store the result in `rag_answer`.

+ Uncomment the `print()` statements to display the result.



In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

rag_prompt = ChatPromptTemplate.from_template(
    """You are a helpful assistant. Use the context below to answer the question.
    Context: {context}
    Question: {question}
    Answer:"""
)

rag_chain = ### YOUR CODE HERE ###

rag_answer = ### YOUR CODE HERE ###

# print(f"Question: What are the main components of LangChain?\n")
# print(f"Answer:\n{rag_answer}")

## Task Group 4 — Building the Agent + RAG Hybrid


### Task 8

The RAG pipeline is ready, but the agent cannot use it directly — agents only interact with tools. To let the agent answer learner questions about the course documentation, we wrap the retriever as a tool. The `@tool` decorator and the `search_documents()` function signature are already provided to support this.

+ Add a docstring that describes when the agent should use this tool.

+ Add a line to invoke `retriever` with `query` and store the result in `docs`.


In [ ]:
@tool
def search_documents(query: str) -> str:
    ### YOUR CODE HERE ###
    docs = retriever.invoke(query)
    return "\n\n".join(doc.page_content for doc in docs)

### Task 9

The platform's learning assistant is almost complete. So far, we have built tools for arithmetic, web search, and document retrieval. We now wire all of them into a single hybrid agent using `hybrid_tools`, a pre-defined list that gives the agent access to all available tools.

+ Pull `"hwchase17/openai-functions-agent"` from LangChain Hub and store it in `prompt`.

+ Create `agent` using `create_tool_calling_agent()`.

+ Wrap it in `AgentExecutor` stored in `agent_executor`, with `verbose` set to `True` and `stream_runnable` set to `False`.

+ Invoke the agent with three test questions.

+ Uncomment the `print()` statements.



In [ ]:
hybrid_tools = [add, subtract, multiply, web_search, search_documents]

prompt = ### YOUR CODE HERE ###
agent = ### YOUR CODE HERE ###
agent_executor = ### YOUR CODE HERE ###

calc_result = agent_executor.invoke({"input": "What is 47 multiplied by 12?"})
doc_result = agent_executor.invoke({"input": "What are the main components of LangChain?"})
web_result = agent_executor.invoke({"input": "What are the latest developments in AI this week?"})

# print("Calculation Result:", calc_result["output"])
# print("\nDocument Result:", doc_result["output"])
# print("\nWeb Search Result:", web_result["output"])